# Zero-shot learning

_Few-shot & Transfer Learning_

**Recognize a class you have ZERO training examples of, by matching the input to a written description of the class.**

Zero-shot learning means classifying into classes you have zero training examples for.

       The trick: instead of showing the model labeled pictures of the new class, you give it a description of the class.

---

This notebook is a practice scaffold for the **Zero-shot learning** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Visualize the data & results

_Can a model classify handwritten 8s and 9s it NEVER saw as labels, just by matching them to a written description of each digit?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import label

digits = load_digits()                       # 1797 real 8x8 handwritten digits
X = digits.data / 16.0
y = digits.target
imgs = digits.images / 16.0

def holes(im):                               # real count of enclosed background regions
    bg = (im < 0.25).astype(int)
    lab, n = label(bg)
    border = set(lab[0, :]) | set(lab[-1, :]) | set(lab[:, 0]) | set(lab[:, -1])
    return sum(1 for c in range(1, n + 1) if c not in border)

def stats(im):                               # interpretable shape statistics
    top, bot = im[:4, :].sum(), im[4:, :].sum()
    left, right = im[:, :4].sum(), im[:, 4:].sum()
    total = im.sum() + 1e-9
    center = im[:, 3:5].sum()                 # central vertical stroke
    com = (im.sum(1) * np.arange(8)).sum() / total   # vertical center of mass
    return np.array([(top - bot) / total, (left - right) / total,
                     center / total, total / 64.0, com / 7.0])

# 6-D attribute vector per image: [holes, top/bottom, left/right, center, ink, com]
feat = np.array([np.concatenate([[holes(imgs[i])], stats(imgs[i])])
                 for i in range(len(imgs))])

# each CLASS description s_k = the mean attribute vector for that class
A = np.array([feat[y == d].mean(0) for d in range(10)])

SEEN, HELD = [0, 1, 2, 3, 4, 5, 6, 7], [8, 9]   # 8 and 9 are NEVER seen as labels
mask = np.isin(y, SEEN)

xsc = StandardScaler().fit(X[mask])              # standardize pixels
asc = StandardScaler().fit(feat[mask])           # standardize attribute space
# g(x): a regressor mapping image -> attribute space, trained ONLY on classes 0-7
reg = Ridge(alpha=1.0).fit(xsc.transform(X[mask]), asc.transform(feat[mask]))

held = np.isin(y, HELD)
emb = reg.predict(xsc.transform(X[held]))        # embed held-out images
yte = y[held]
S = asc.transform(A)                             # class descriptions in same space

def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

# zero-shot rule: y_hat = argmax_k cos(g(x), s_k) over the unseen classes {8, 9}
pred = np.array([HELD[int(np.argmax([cos(v, S[c]) for c in HELD]))] for v in emb])
acc = (pred == yte).mean()                       # 0.814 -> 81.4% on unseen 8 vs 9

# plot each held-out image by (similarity to 8-desc, similarity to 9-desc)
import matplotlib.pyplot as plt
x8 = [cos(v, S[8]) for v in emb]
x9 = [cos(v, S[9]) for v in emb]
plt.scatter(np.array(x8)[yte == 8], np.array(x9)[yte == 8], color="#4ea1ff", label="true 8")
plt.scatter(np.array(x8)[yte == 9], np.array(x9)[yte == 9], color="#7ee787", label="true 9")
plt.plot([-0.5, 1], [-0.5, 1], "--", color="gray")
plt.xlabel("cosine similarity to the 8-description")
plt.ylabel("cosine similarity to the 9-description")
plt.title("Held-out digits 8 and 9 land nearest their own attribute description (zero-shot)")
plt.legend()
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Three class descriptions: $s_{\text{cat}} = [1, 0]$, $s_{\text{dog}} = [0, 1]$, $s_{\text{wolf}} = [0, 1]$ (a wolf is described like a dog). A new image embeds to $g(x) = [0.1, 0.9]$. Which class wins, and why might cat lose badly?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Cosine to cat $[1,0]$: $\frac{0.1\cdot1 + 0.9\cdot0}{\sqrt{0.1^2+0.9^2}\cdot 1} = \frac{0.1}{0.906} \approx 0.11$. — _Only the first attribute overlaps, and the input is tiny there._
- Cosine to dog $[0,1]$: $\frac{0.1\cdot0 + 0.9\cdot1}{0.906\cdot 1} = \frac{0.9}{0.906} \approx 0.99$. — _The input points almost entirely along the second attribute, same as dog._
- Cosine to wolf $[0,1]$: same as dog, $\approx 0.99$. — _Wolf and dog share the same description, so they tie._
- Take the $\arg\max$: dog (and wolf) at $0.99$ beat cat at $0.11$. — _The prediction is the class whose description vector points most like the input._

**Answer:** Dog and wolf tie at cosine $\approx 0.99$; cat loses with $\approx 0.11$ because the input barely points along cat's attribute. Identical descriptions (dog vs wolf) cannot be told apart — zero-shot needs distinct descriptions.

</details>

**Problem 2.** Explain in one or two sentences how zero-shot learning is the $K=0$ case of few-shot learning.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Few-shot builds a class prototype by averaging $K$ support examples: $c_k = \frac{1}{K}\sum_i g(x_i)$. — _That average is the class's stand-in vector._
- At $K=0$ there are no examples to average, so replace the prototype with a description embedding $s_k$. — _The written description fills the role the examples would have played._

**Answer:** Few-shot classifies by nearest prototype (an average of $K$ examples). Zero-shot sets $K=0$ and swaps the unavailable prototype for a class description $s_k$, then classifies by nearest description — same rule, different source for the class vector.

</details>